# LSTM sobre fusão temporal (local + GPS + EKF2)

**Pré-requisito lógico:** `Infos_fusao_gps_local_ekf.ipynb` (merge `vehicle_local_position` + `vehicle_gps_position` + `ekf2_innovations`).

**Objetivo:** classificar o **cenário experimental** (`Normal` / `GPS Spoofing` / `Ping DoS`) a partir de **janelas multivariadas** de sensores alinhados.

**Split:** 80% / 20% **temporal dentro de cada voo** (sem misturar início e fim do mesmo log), depois concatenação dos conjuntos — reduz vazamento “vizinho a vizinho” dentro do mesmo ficheiro em relação a um split puramente aleatório.


## Síntese do que os dados mostram (antes do modelo)

Com base no `merge_asof` e nas séries já vistas nos notebooks anteriores:

| Sinal | Normal vs Ping DoS vs Spoofing |
|-------|----------------------------------|
| **Cadência `dt`** | ~**0,10 s** em todos — reflete o logger/SITL, **não** discrimina ataque. |
| **`dv_xy`** (módulo de **v_xy_loc − v_xy_gps**) | **Medianas ~0,037–0,038 m/s** muito parecidas; **máximos** ~0,77–0,88 m/s — pouco para separar **três classes** sozinho. |
| **`innov_norm` (‖vel_pos_innov[0:2]‖)** | **Medianas ~0,067** em todos; cauda global (`>p95`) ~**5%** em cada classe — **não** é um “alarme” trivial por limiar fixo. |
| **`eph_loc` / `eph_gps`** | Distribuições **muito sobrepostas** entre classes neste export. |
| **`r_xy`** (distância horizontal ao primeiro ponto do voo) | **Máximo ~3×10² m** (Normal / DoS) vs **~7,9×10⁴ m** (Spoofing) — **principal separador** do spoofing em relação aos outros dois. |

**Conclusão para o TCC:** o classificador precisa de **features que capturem trajetória** (`x`, `y`, acumulado ou `r_xy`) e, secundariamente, **inovações / discrepâncias**; esperar **confusão Normal ↔ Ping DoS** se o modelo depender sobretudo de ritmo ou de `eph`. O notebook abaixo inclui **`x`, `y`, `z`** e **`r_xy`** nas *features* por janela para refletir isso.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

def find_px4_quad_sitl() -> Path:
    'Sobe a partir do cwd até achar PX4-QUAD-SITL.'
    here = Path.cwd().resolve()
    for anchor in [here, *here.parents]:
        p = anchor / "data" / "UAVAttackData" / "Simulated - OTU Survey" / "PX4-QUAD-SITL"
        if p.is_dir():
            return p
    raise FileNotFoundError("data/UAVAttackData/.../PX4-QUAD-SITL")


BASE_PATH = find_px4_quad_sitl()
NORMAL, SPOOFING, DOS = BASE_PATH / "Normal", BASE_PATH / "GPS Spoofing", BASE_PATH / "Ping DoS"

LABEL_MAP = {"Normal": 0, "GPS Spoofing": 1, "Ping DoS": 2}


def pick_vehicle_local_position_csv(folder: Path) -> Path:
    matches = sorted(folder.glob("*vehicle_local_position*.csv"))
    matches = [m for m in matches if "setpoint" not in m.name and "groundtruth" not in m.name]
    if not matches:
        raise FileNotFoundError(folder)
    return matches[0]


def log_prefix(folder: Path) -> str:
    name = pick_vehicle_local_position_csv(folder).name
    suf = "_vehicle_local_position_0.csv"
    if not name.endswith(suf):
        raise ValueError(name)
    return name[: -len(suf)]


def load_merged(folder: Path, condition: str) -> pd.DataFrame:
    pref = log_prefix(folder)
    loc = pd.read_csv(folder / f"{pref}_vehicle_local_position_0.csv", sep=",").sort_values("timestamp")
    gps = pd.read_csv(folder / f"{pref}_vehicle_gps_position_0.csv", sep=",").sort_values("timestamp")
    ekf = pd.read_csv(folder / f"{pref}_ekf2_innovations_0.csv", sep=",").sort_values("timestamp")
    m = pd.merge_asof(loc, gps, on="timestamp", suffixes=("_loc", "_gps"), direction="backward")
    m = pd.merge_asof(m.sort_values("timestamp"), ekf.sort_values("timestamp"), on="timestamp", direction="backward")
    return m.assign(condition=condition)


merged = pd.concat(
    [
        load_merged(NORMAL, "Normal"),
        load_merged(SPOOFING, "GPS Spoofing"),
        load_merged(DOS, "Ping DoS"),
    ],
    ignore_index=True,
).copy()

cols_innov = ["vel_pos_innov[0]", "vel_pos_innov[1]", "vel_pos_innov[2]"]
M = merged[cols_innov].astype(float).to_numpy()
merged["innov_norm"] = np.sqrt(np.sum(M * M, axis=1))
merged["v_xy_gps"] = np.sqrt(merged["vel_n_m_s"].astype(float) ** 2 + merged["vel_e_m_s"].astype(float) ** 2)
merged["v_xy_loc"] = np.sqrt(merged["vx"].astype(float) ** 2 + merged["vy"].astype(float) ** 2)
merged["dv_xy"] = np.abs(merged["v_xy_loc"] - merged["v_xy_gps"])


def _r_xy_series(x, y):
    x0, y0 = float(x.iloc[0]), float(y.iloc[0])
    return np.sqrt((x - x0) ** 2 + (y - y0) ** 2)


merged["r_xy"] = 0.0
for _cond in merged["condition"].unique():
    _ix = merged["condition"] == _cond
    merged.loc[_ix, "r_xy"] = _r_xy_series(merged.loc[_ix, "x"], merged.loc[_ix, "y"]).to_numpy(dtype=float)
merged["y_label"] = merged["condition"].map(LABEL_MAP)

print(merged.shape)
merged.groupby("condition").size()


(44957, 113)


/var/folders/8_/_x4q463d1yl_x911vq7_v8kr0000gn/T/ipykernel_15519/2396537755.py:45: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  return m.assign(condition=condition)


condition
GPS Spoofing    15453
Normal          14765
Ping DoS        14739
dtype: int64

## 1. Features e hiperparâmetros

Lista fixa de colunas numéricas por passo de tempo; `WINDOW` e fração de treino ajustáveis.


In [2]:
FEATS = [
    "x",
    "y",
    "z",
    "vx",
    "vy",
    "vz",
    "eph_loc",
    "eph_gps",
    "innov_norm",
    "dv_xy",
    "r_xy",
]
WINDOW = 40
FRAC_TRAIN = 0.8

missing = [c for c in FEATS if c not in merged.columns]
if missing:
    raise ValueError("Colunas em falta: " + str(missing))

data = merged[FEATS + ["condition", "y_label", "timestamp"]].replace([np.inf, -np.inf], np.nan).dropna()
print("linhas após dropna:", len(data))


linhas após dropna: 44957


## 2. Janelas deslizantes + partição temporal por voo

Cada `condition` corresponde a **um** ficheiro de voo: o índice temporal ordenado define treino (primeiros 80%) e teste (últimos 20%).


In [3]:
def build_windows_for_condition(df: pd.DataFrame, window: int, frac_train: float):
    df = df.sort_values("timestamp").reset_index(drop=True)
    n = len(df)
    split = int(frac_train * n)
    split = max(split, window + 1)
    X_train, y_train = [], []
    X_test, y_test = [], []
    label = int(df["y_label"].iloc[0])
    for i in range(0, split - window):
        X_train.append(df.iloc[i : i + window][FEATS].to_numpy(dtype=np.float32))
        y_train.append(label)
    for i in range(split, n - window):
        X_test.append(df.iloc[i : i + window][FEATS].to_numpy(dtype=np.float32))
        y_test.append(label)
    return np.stack(X_train), np.array(y_train), np.stack(X_test), np.array(y_test)


X_tr_parts, y_tr_parts, X_te_parts, y_te_parts = [], [], [], []
for cond in ["Normal", "GPS Spoofing", "Ping DoS"]:
    sub = data.loc[data["condition"] == cond].copy()
    Xt, yt, Xe, ye = build_windows_for_condition(sub, WINDOW, FRAC_TRAIN)
    X_tr_parts.append(Xt)
    y_tr_parts.append(yt)
    X_te_parts.append(Xe)
    y_te_parts.append(ye)
    print(cond, "train windows", len(yt), "test windows", len(ye))

X_train = np.concatenate(X_tr_parts, axis=0)
y_train = np.concatenate(y_tr_parts)
X_test = np.concatenate(X_te_parts, axis=0)
y_test = np.concatenate(y_te_parts)
print("X_train", X_train.shape, "X_test", X_test.shape)


Normal train windows 11772 test windows 2913
GPS Spoofing train windows 12322 test windows 3051
Ping DoS train windows 11751 test windows 2908
X_train (35845, 40, 11) X_test (8872, 40, 11)


## 3. Normalização (só no treino)

`StandardScaler` ajustado aos passos do treino, aplicado igualmente a treino e teste.


In [4]:
from sklearn.preprocessing import StandardScaler

ns, win, nf = X_train.shape
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.reshape(-1, nf)).reshape(ns, win, nf)
X_test_s = scaler.transform(X_test.reshape(-1, nf)).reshape(X_test.shape[0], win, nf)


## 4. LSTM (Keras) + relatório

Rede pequena para prototipagem rápida; aumente `WINDOW` / unidades se o TCC exigir mais capacidade.


In [5]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report

tf.random.set_seed(42)
np.random.seed(42)

model = keras.Sequential(
    [
        layers.Input(shape=(WINDOW, nf)),
        layers.LSTM(48, return_sequences=False),
        layers.Dropout(0.25),
        layers.Dense(3, activation="softmax"),
    ]
)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(
    X_train_s,
    y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1,
)

y_pred = np.argmax(model.predict(X_test_s, verbose=0), axis=1)
names = ["Normal", "GPS Spoofing", "Ping DoS"]
print(classification_report(y_test, y_pred, target_names=names, digits=3))


Epoch 1/15
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7673 - loss: 0.4448 - val_accuracy: 0.2594 - val_loss: 2.8738
Epoch 2/15
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8576 - loss: 0.2699 - val_accuracy: 0.3735 - val_loss: 2.7845
Epoch 3/15
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8707 - loss: 0.2374 - val_accuracy: 0.3464 - val_loss: 3.3209
Epoch 4/15
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8800 - loss: 0.2156 - val_accuracy: 0.3768 - val_loss: 3.2622
Epoch 5/15
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8932 - loss: 0.1931 - val_accuracy: 0.5300 - val_loss: 2.6507
Epoch 6/15
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8963 - loss: 0.1827 - val_accuracy: 0.4382 - val_loss: 3.5397
Epoch 7/15
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9055 - loss: 0.1680 - val_accuracy: 0.3434 - val_loss: 4.4065
Epoch 8/15
505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9095 - loss: 0.1672 - val_accuracy: 0.

## 5. Limitações (para escrever no TCC)

- Só há **um voo por classe**; o split é **temporal dentro do mesmo ficheiro**, não “voo novo nunca visto”.
- **Normal** e **Ping DoS** podem ser **estatisticamente parecidos** em muitas features; o **spoofing** tende a destacar-se por **deriva horizontal** — ver se a matriz de confusão reflete isso.
- Comparar com **RandomForest** nas mesmas janelas achatadas (`win * n_features`) pode ser um baseline útil em notebook adicional.
